# Transformer Neural Machine Translation (Full Pipeline)

An end-to-end English→German NMT pipeline built from the original *Attention is All You Need* paper.

**Pipeline stages:**
1. Configuration
2. Data preprocessing (Multi30k EN-DE)
3. BPE tokenizer training (shared vs. separate)
4. Dataset & DataLoader
5. Transformer model (Encoder – Decoder w/ cross-attention)
6. Training loop (Noam scheduler + label smoothing + teacher forcing)
7. Evaluation (greedy / beam search + BLEU)
8. Visualization (loss curves + attention heatmaps)
9. Smoke-test demo on a tiny synthetic batch

## 0. Install dependencies (uncomment if needed)

In [ ]:
# !pip install torch torchtext datasets tokenizers sacrebleu numpy tqdm matplotlib seaborn

## 1. Configuration

In [ ]:
import torch

class TransformerConfig:
    """Configuration parameters for the Transformer model."""

    # Model architecture
    d_model = 512
    num_heads = 8
    num_encoder_layers = 6
    num_decoder_layers = 6
    d_ff = 2048
    dropout = 0.1
    max_seq_length = 100

    # Tokenizer
    vocab_size = 10000
    use_shared_vocab = False

    # Training
    batch_size = 64
    num_epochs = 20
    learning_rate = 1e-4
    warmup_steps = 4000
    label_smoothing = 0.1

    # Data
    src_lang = 'en'
    tgt_lang = 'de'

    # Special tokens
    PAD_TOKEN = '<pad>'
    UNK_TOKEN = '<unk>'
    SOS_TOKEN = '<sos>'
    EOS_TOKEN = '<eos>'

    # Paths
    data_dir = './data'
    model_dir = './models'
    tokenizer_dir = './tokenizers'

    @staticmethod
    def get_device():
        return 'cuda' if torch.cuda.is_available() else 'cpu'


config = TransformerConfig()
print('Device:', config.get_device())

## 2. Data Preprocessing

Loads the Multi30k EN-DE parallel corpus from HuggingFace and applies:
- Unicode NFKC normalization
- Punctuation spacing
- Lowercasing
- Length & ratio filtering

In [ ]:
import re
import unicodedata
from typing import List, Tuple


class BilingualPreprocessor:
    """Preprocessor for bilingual parallel corpora."""

    def __init__(self, src_lang: str, tgt_lang: str):
        self.src_lang = src_lang
        self.tgt_lang = tgt_lang

    def normalize_unicode(self, text: str) -> str:
        return unicodedata.normalize('NFKC', text)

    def clean_punctuation(self, text: str, lang: str) -> str:
        text = self.normalize_unicode(text)
        text = re.sub(r'\s+', ' ', text)
        text = re.sub(r'([.,!?;:])', r' \1 ', text)
        text = re.sub(r'\s+', ' ', text)
        return text.strip()

    def normalize_case(self, text: str, lowercase: bool = True) -> str:
        return text.lower() if lowercase else text

    def preprocess_sentence(self, text: str, lang: str, lowercase: bool = True) -> str:
        text = text.strip()
        text = self.normalize_unicode(text)
        text = self.clean_punctuation(text, lang)
        text = self.normalize_case(text, lowercase)
        return text

    def preprocess_parallel_corpus(
        self,
        src_sentences: List[str],
        tgt_sentences: List[str],
        max_length: int = 100,
    ) -> Tuple[List[str], List[str]]:
        assert len(src_sentences) == len(tgt_sentences)
        out_src, out_tgt = [], []
        for src, tgt in zip(src_sentences, tgt_sentences):
            s = self.preprocess_sentence(src, self.src_lang)
            t = self.preprocess_sentence(tgt, self.tgt_lang)
            if not s or not t:
                continue
            s_tok, t_tok = s.split(), t.split()
            if len(s_tok) > max_length or len(t_tok) > max_length:
                continue
            ratio = len(s_tok) / len(t_tok) if t_tok else 0
            if ratio < 0.5 or ratio > 2.0:
                continue
            out_src.append(s)
            out_tgt.append(t)
        print(f'Original pairs: {len(src_sentences)}  |  After filtering: {len(out_src)}')
        return out_src, out_tgt


def load_and_preprocess_multi30k(split: str = 'train', src_lang: str = 'en', tgt_lang: str = 'de'):
    """Download and preprocess the Multi30k dataset (requires internet)."""
    from datasets import load_dataset
    print(f'Loading Multi30k ({split})...')
    dataset = load_dataset('bentrevett/multi30k', split=split)
    src_sentences = [ex[src_lang] for ex in dataset]
    tgt_sentences = [ex[tgt_lang] for ex in dataset]
    return BilingualPreprocessor(src_lang, tgt_lang).preprocess_parallel_corpus(
        src_sentences, tgt_sentences, max_length=100
    )


# Demo: small synthetic example (does not require downloading data)
demo_preprocessor = BilingualPreprocessor('en', 'de')
print(demo_preprocessor.preprocess_sentence('Hello, World!', 'en'))

## 3. BPE Tokenizer Training

Two strategies are supported:
- **Shared vocabulary** — single BPE model trained on `src + tgt` (recommended for EN-DE since both use Latin script and share cognates).
- **Separate vocabularies** — independent BPE models for source and target (better for distant language pairs).

In [ ]:
import os
from typing import Optional


class BilingualTokenizerTrainer:
    def __init__(self, vocab_size: int = 10000, min_frequency: int = 2, special_tokens=None):
        self.vocab_size = vocab_size
        self.min_frequency = min_frequency
        self.special_tokens = special_tokens or ['<pad>', '<unk>', '<sos>', '<eos>']

    def create_tokenizer(self):
        from tokenizers import Tokenizer, models, pre_tokenizers, decoders
        from tokenizers.normalizers import Lowercase, NFKC, Sequence as NormSequence
        tok = Tokenizer(models.BPE(unk_token='<unk>'))
        tok.normalizer = NormSequence([NFKC(), Lowercase()])
        tok.pre_tokenizer = pre_tokenizers.Whitespace()
        tok.decoder = decoders.BPE()
        return tok

    def _post_process(self, tokenizer):
        from tokenizers import processors
        tokenizer.post_processor = processors.TemplateProcessing(
            single='<sos> $A <eos>',
            special_tokens=[
                ('<sos>', tokenizer.token_to_id('<sos>')),
                ('<eos>', tokenizer.token_to_id('<eos>')),
            ],
        )

    def train_shared_tokenizer(self, src_sentences, tgt_sentences, save_path):
        from tokenizers import trainers
        tokenizer = self.create_tokenizer()
        trainer = trainers.BpeTrainer(
            vocab_size=self.vocab_size,
            min_frequency=self.min_frequency,
            special_tokens=self.special_tokens,
        )
        tokenizer.train_from_iterator(src_sentences + tgt_sentences, trainer=trainer)
        self._post_process(tokenizer)
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        tokenizer.save(save_path)
        print(f'Shared tokenizer saved → {save_path} | vocab={len(tokenizer.get_vocab())}')
        return tokenizer

    def train_separate_tokenizers(self, src_sentences, tgt_sentences, src_save_path, tgt_save_path):
        from tokenizers import trainers
        outputs = []
        for sentences, path in [(src_sentences, src_save_path), (tgt_sentences, tgt_save_path)]:
            tok = self.create_tokenizer()
            trainer = trainers.BpeTrainer(
                vocab_size=self.vocab_size,
                min_frequency=self.min_frequency,
                special_tokens=self.special_tokens,
            )
            tok.train_from_iterator(sentences, trainer=trainer)
            self._post_process(tok)
            os.makedirs(os.path.dirname(path), exist_ok=True)
            tok.save(path)
            print(f'Tokenizer saved → {path} | vocab={len(tok.get_vocab())}')
            outputs.append(tok)
        return outputs[0], outputs[1]


def load_tokenizer(path: str):
    from tokenizers import Tokenizer
    return Tokenizer.from_file(path)

## 4. Dataset & DataLoader

Pairs source/target sentences, tokenizes them, and prepares decoder inputs / outputs for teacher forcing:
- `tgt_input  = <sos> w1 w2 w3` (decoder input)
- `tgt_output = w1 w2 w3 <eos>` (loss target)

In [ ]:
from torch.utils.data import Dataset, DataLoader


class TranslationDataset(Dataset):
    def __init__(self, src_sentences, tgt_sentences, src_tokenizer, tgt_tokenizer, max_length: int = 100):
        assert len(src_sentences) == len(tgt_sentences)
        self.src_sentences = src_sentences
        self.tgt_sentences = tgt_sentences
        self.src_tokenizer = src_tokenizer
        self.tgt_tokenizer = tgt_tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.src_sentences)

    def __getitem__(self, idx):
        src_ids = self.src_tokenizer.encode(self.src_sentences[idx]).ids
        tgt_ids = self.tgt_tokenizer.encode(self.tgt_sentences[idx]).ids
        if len(src_ids) > self.max_length:
            src_ids = src_ids[: self.max_length - 1] + [src_ids[-1]]
        if len(tgt_ids) > self.max_length:
            tgt_ids = tgt_ids[: self.max_length - 1] + [tgt_ids[-1]]
        return {
            'src': torch.tensor(src_ids, dtype=torch.long),
            'tgt_input': torch.tensor(tgt_ids[:-1], dtype=torch.long),
            'tgt_output': torch.tensor(tgt_ids[1:], dtype=torch.long),
        }


def collate_fn(batch, src_pad_id=0, tgt_pad_id=0):
    src = [b['src'] for b in batch]
    tin = [b['tgt_input'] for b in batch]
    tout = [b['tgt_output'] for b in batch]
    return {
        'src': torch.nn.utils.rnn.pad_sequence(src, batch_first=True, padding_value=src_pad_id),
        'tgt_input': torch.nn.utils.rnn.pad_sequence(tin, batch_first=True, padding_value=tgt_pad_id),
        'tgt_output': torch.nn.utils.rnn.pad_sequence(tout, batch_first=True, padding_value=tgt_pad_id),
    }


def create_dataloaders(train_src, train_tgt, val_src, val_tgt, src_tokenizer, tgt_tokenizer,
                       batch_size: int = 64, max_length: int = 100, num_workers: int = 0):
    src_pad_id = src_tokenizer.token_to_id('<pad>')
    tgt_pad_id = tgt_tokenizer.token_to_id('<pad>')
    train_ds = TranslationDataset(train_src, train_tgt, src_tokenizer, tgt_tokenizer, max_length)
    val_ds = TranslationDataset(val_src, val_tgt, src_tokenizer, tgt_tokenizer, max_length)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers,
                              collate_fn=lambda b: collate_fn(b, src_pad_id, tgt_pad_id), pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers,
                            collate_fn=lambda b: collate_fn(b, src_pad_id, tgt_pad_id), pin_memory=True)
    return train_loader, val_loader

## 5. Transformer Model

**Components:**
- Sinusoidal positional encoding
- Multi-head (self & cross) attention
- Position-wise feed-forward (FFN)
- Encoder / Decoder stacks
- Padding & causal masking

In [ ]:
import math
import torch.nn as nn
import torch.nn.functional as F


class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, : x.size(1), :]
        return self.dropout(x)


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        return torch.matmul(attn, V), attn

    def forward(self, query, key, value, mask=None):
        B = query.size(0)
        Q = self.W_q(query).view(B, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(key).view(B, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(value).view(B, -1, self.num_heads, self.d_k).transpose(1, 2)
        out, _ = self.scaled_dot_product_attention(Q, K, V, mask)
        out = out.transpose(1, 2).contiguous().view(B, -1, self.d_model)
        return self.W_o(out)


class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))


class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, src_mask=None):
        x = self.norm1(x + self.dropout1(self.self_attention(x, x, x, src_mask)))
        x = self.norm2(x + self.dropout2(self.feed_forward(x)))
        return x


class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers, num_heads, d_ff, max_seq_length, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_seq_length, dropout)
        self.layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.dropout = nn.Dropout(dropout)

    def forward(self, src, src_mask=None):
        x = self.embedding(src) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        for layer in self.layers:
            x = layer(x, src_mask)
        return x


class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        x = self.norm1(x + self.dropout1(self.self_attention(x, x, x, tgt_mask)))
        x = self.norm2(x + self.dropout2(self.cross_attention(x, encoder_output, encoder_output, src_mask)))
        x = self.norm3(x + self.dropout3(self.feed_forward(x)))
        return x


class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers, num_heads, d_ff, max_seq_length, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_seq_length, dropout)
        self.layers = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.dropout = nn.Dropout(dropout)

    def forward(self, tgt, encoder_output, src_mask=None, tgt_mask=None):
        x = self.embedding(tgt) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        for layer in self.layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)
        return x


class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, num_encoder_layers=6,
                 num_decoder_layers=6, num_heads=8, d_ff=2048, max_seq_length=100,
                 dropout=0.1, pad_idx=0):
        super().__init__()
        self.encoder = Encoder(src_vocab_size, d_model, num_encoder_layers, num_heads, d_ff, max_seq_length, dropout)
        self.decoder = Decoder(tgt_vocab_size, d_model, num_decoder_layers, num_heads, d_ff, max_seq_length, dropout)
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)
        self.pad_idx = pad_idx

    def make_src_mask(self, src):
        return (src != self.pad_idx).unsqueeze(1).unsqueeze(2)

    def make_tgt_mask(self, tgt):
        _, tgt_len = tgt.size()
        pad_mask = (tgt != self.pad_idx).unsqueeze(1).unsqueeze(2)
        sub_mask = torch.tril(torch.ones((tgt_len, tgt_len), device=tgt.device)).bool()
        return pad_mask & sub_mask

    def forward(self, src, tgt):
        src_mask = self.make_src_mask(src)
        tgt_mask = self.make_tgt_mask(tgt)
        enc = self.encoder(src, src_mask)
        dec = self.decoder(tgt, enc, src_mask, tgt_mask)
        return self.fc_out(dec)

## 6. Training Loop

- **Noam Scheduler:** `lr = d_model^(-0.5) * min(step^(-0.5), step * warmup^(-1.5))`
- **Label Smoothing Loss:** soft targets `(1-ε)` on the true class, `ε/(V-2)` on others
- **Teacher Forcing:** decoder consumes ground-truth tokens during training
- Gradient clipping at norm 1.0

In [ ]:
import time
import json
import torch.optim as optim

try:
    from tqdm import tqdm
except ImportError:
    def tqdm(x, **k):  # type: ignore
        return x


class NoamScheduler:
    def __init__(self, optimizer, d_model: int, warmup_steps: int = 4000, factor: float = 1.0):
        self.optimizer = optimizer
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        self.factor = factor
        self.step_num = 0

    def step(self):
        self.step_num += 1
        lr = self.get_lr()
        for pg in self.optimizer.param_groups:
            pg['lr'] = lr

    def get_lr(self):
        s = self.step_num if self.step_num > 0 else 1
        return self.factor * (self.d_model ** -0.5) * min(s ** -0.5, s * self.warmup_steps ** -1.5)


class LabelSmoothingLoss(nn.Module):
    def __init__(self, vocab_size: int, pad_idx: int, smoothing: float = 0.1):
        super().__init__()
        self.vocab_size = vocab_size
        self.pad_idx = pad_idx
        self.smoothing = smoothing
        self.confidence = 1.0 - smoothing

    def forward(self, pred, target):
        true_dist = torch.zeros_like(pred)
        true_dist.fill_(self.smoothing / (self.vocab_size - 2))
        true_dist.scatter_(1, target.unsqueeze(1), self.confidence)
        true_dist[:, self.pad_idx] = 0
        mask = target == self.pad_idx
        if mask.any():
            true_dist[mask] = 0
        loss = -(true_dist * torch.log_softmax(pred, dim=1)).sum(dim=1)
        loss = loss.masked_fill(mask, 0)
        return loss.sum() / (~mask).sum().clamp(min=1)


class Trainer:
    def __init__(self, model, train_loader, val_loader, optimizer, scheduler, criterion,
                 device: str = 'cuda', checkpoint_dir: str = './checkpoints'):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.criterion = criterion
        self.device = device
        self.checkpoint_dir = checkpoint_dir
        os.makedirs(checkpoint_dir, exist_ok=True)
        self.train_losses, self.val_losses, self.learning_rates = [], [], []

    def train_epoch(self, epoch: int):
        self.model.train()
        total = 0.0
        pbar = tqdm(self.train_loader, desc=f'Epoch {epoch}')
        for batch in pbar:
            src = batch['src'].to(self.device)
            tin = batch['tgt_input'].to(self.device)
            tout = batch['tgt_output'].to(self.device)
            logits = self.model(src, tin)
            B, T, V = logits.shape
            loss = self.criterion(logits.reshape(-1, V), tout.reshape(-1))
            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.optimizer.step()
            self.scheduler.step()
            total += loss.item()
        avg = total / max(1, len(self.train_loader))
        self.train_losses.append(avg)
        self.learning_rates.append(self.scheduler.get_lr())
        return avg

    @torch.no_grad()
    def validate(self):
        self.model.eval()
        total = 0.0
        for batch in tqdm(self.val_loader, desc='Validation'):
            src = batch['src'].to(self.device)
            tin = batch['tgt_input'].to(self.device)
            tout = batch['tgt_output'].to(self.device)
            logits = self.model(src, tin)
            B, T, V = logits.shape
            loss = self.criterion(logits.reshape(-1, V), tout.reshape(-1))
            total += loss.item()
        avg = total / max(1, len(self.val_loader))
        self.val_losses.append(avg)
        return avg

    def train(self, num_epochs: int):
        best = float('inf')
        for epoch in range(1, num_epochs + 1):
            t0 = time.time()
            tl = self.train_epoch(epoch)
            vl = self.validate()
            print(f'Epoch {epoch}: train={tl:.4f}  val={vl:.4f}  lr={self.scheduler.get_lr():.6f}  ({time.time() - t0:.1f}s)')
            if vl < best:
                best = vl
                self.save_checkpoint(epoch, vl, is_best=True)
        self.save_history()

    def save_checkpoint(self, epoch, val_loss, is_best=False):
        path = os.path.join(self.checkpoint_dir, 'best_model.pt' if is_best else f'checkpoint_epoch_{epoch}.pt')
        torch.save({
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_step': self.scheduler.step_num,
            'val_loss': val_loss,
            'train_losses': self.train_losses,
            'val_losses': self.val_losses,
            'learning_rates': self.learning_rates,
        }, path)

    def save_history(self):
        path = os.path.join(self.checkpoint_dir, 'training_history.json')
        with open(path, 'w') as f:
            json.dump({
                'train_losses': self.train_losses,
                'val_losses': self.val_losses,
                'learning_rates': self.learning_rates,
            }, f, indent=2)

## 7. Evaluation: Greedy / Beam Search + BLEU

In [ ]:
class TranslationGenerator:
    def __init__(self, model, src_tokenizer, tgt_tokenizer, device='cuda'):
        self.model = model.to(device).eval()
        self.src_tokenizer = src_tokenizer
        self.tgt_tokenizer = tgt_tokenizer
        self.device = device
        self.pad_id = tgt_tokenizer.token_to_id('<pad>')
        self.sos_id = tgt_tokenizer.token_to_id('<sos>')
        self.eos_id = tgt_tokenizer.token_to_id('<eos>')

    @torch.no_grad()
    def greedy_search(self, src_text: str, max_length: int = 100):
        src_ids = torch.tensor([self.src_tokenizer.encode(src_text).ids], dtype=torch.long, device=self.device)
        src_mask = self.model.make_src_mask(src_ids)
        enc = self.model.encoder(src_ids, src_mask)
        tgt_ids = [self.sos_id]
        for _ in range(max_length):
            tgt = torch.tensor([tgt_ids], dtype=torch.long, device=self.device)
            tgt_mask = self.model.make_tgt_mask(tgt)
            dec = self.model.decoder(tgt, enc, src_mask, tgt_mask)
            nxt = self.model.fc_out(dec[:, -1, :]).argmax(-1).item()
            tgt_ids.append(nxt)
            if nxt == self.eos_id:
                break
        return self.tgt_tokenizer.decode(tgt_ids), tgt_ids

    @torch.no_grad()
    def beam_search(self, src_text: str, beam_width: int = 5, max_length: int = 100, length_penalty: float = 0.6):
        src_ids = torch.tensor([self.src_tokenizer.encode(src_text).ids], dtype=torch.long, device=self.device)
        src_mask = self.model.make_src_mask(src_ids)
        enc = self.model.encoder(src_ids, src_mask)
        beams = [([self.sos_id], 0.0)]
        for _ in range(max_length):
            cands = []
            for seq, score in beams:
                if seq[-1] == self.eos_id:
                    cands.append((seq, score))
                    continue
                tgt = torch.tensor([seq], dtype=torch.long, device=self.device)
                tgt_mask = self.model.make_tgt_mask(tgt)
                dec = self.model.decoder(tgt, enc, src_mask, tgt_mask)
                lp = F.log_softmax(self.model.fc_out(dec[:, -1, :]), dim=-1)
                topk_lp, topk_idx = lp.topk(beam_width)
                for i in range(beam_width):
                    cands.append((seq + [topk_idx[0, i].item()], score + topk_lp[0, i].item()))
            scored = [(s, sc, sc / (len(s) ** length_penalty)) for s, sc in cands]
            scored.sort(key=lambda x: x[2], reverse=True)
            beams = [(s, sc) for s, sc, _ in scored[:beam_width]]
            if all(s[-1] == self.eos_id for s, _ in beams):
                break
        best_seq, best_score = beams[0]
        return self.tgt_tokenizer.decode(best_seq), best_seq, best_score

    def translate_batch(self, src_texts, method='greedy', beam_width=5):
        out = []
        for s in src_texts:
            if method == 'greedy':
                t, _ = self.greedy_search(s)
            else:
                t, _, _ = self.beam_search(s, beam_width=beam_width)
            out.append(t)
        return out


def compute_bleu(hypotheses, references):
    from sacrebleu import corpus_bleu
    refs = [[r] for r in references]
    bleu = corpus_bleu(hypotheses, list(zip(*refs)))
    return {'bleu': bleu.score, 'precisions': bleu.precisions, 'bp': bleu.bp,
            'sys_len': bleu.sys_len, 'ref_len': bleu.ref_len}

## 8. Visualization (loss curves + attention)

In [ ]:
def plot_training_history(history_path: str = './checkpoints/training_history.json'):
    import matplotlib.pyplot as plt
    if not os.path.exists(history_path):
        print(f'History file not found: {history_path}')
        return
    with open(history_path) as f:
        h = json.load(f)
    epochs = range(1, len(h['train_losses']) + 1)
    fig, ax = plt.subplots(2, 1, figsize=(12, 8))
    ax[0].plot(epochs, h['train_losses'], 'o-', label='Train')
    ax[0].plot(epochs, h['val_losses'], 's-', label='Val')
    ax[0].set_xlabel('Epoch'); ax[0].set_ylabel('Loss'); ax[0].set_title('Loss')
    ax[0].legend(); ax[0].grid(True, alpha=0.3)
    ax[1].plot(epochs, h['learning_rates'], 'o-', color='green')
    ax[1].set_xlabel('Epoch'); ax[1].set_ylabel('LR'); ax[1].set_title('Learning Rate')
    ax[1].grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()


def plot_attention_heatmap(attention_weights, src_tokens, tgt_tokens, title='Attention'):
    import matplotlib.pyplot as plt
    import numpy as np
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(attention_weights, cmap='viridis', aspect='auto')
    ax.set_xticks(np.arange(len(src_tokens))); ax.set_yticks(np.arange(len(tgt_tokens)))
    ax.set_xticklabels(src_tokens, rotation=45, ha='right'); ax.set_yticklabels(tgt_tokens)
    ax.set_xlabel('Source'); ax.set_ylabel('Target'); ax.set_title(title)
    fig.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

## Visual Walkthrough — Concepts for the Presentation

The cells below render diagrams that explain the Transformer step-by-step. Run them in order to produce slide-ready figures.


### Figure 1 — Transformer Architecture (encoder + decoder + cross-attention)

In [ ]:

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(11, 8))
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis("off")

def box(x, y, w, h, label, color):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.08",
                                linewidth=1.5, facecolor=color, edgecolor="black"))
    ax.text(x + w/2, y + h/2, label, ha="center", va="center", fontsize=9, fontweight="bold")

def arrow(x1, y1, x2, y2, color="black"):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle="->",
                                 mutation_scale=15, color=color, linewidth=1.6))

# Encoder stack (left)
box(0.5, 0.4, 3.0, 0.8, "Source tokens  →  Embedding + Positional Enc.", "#FFE5B4")
box(0.5, 1.6, 3.0, 1.2, "Multi-Head Self-Attention\n+ Add & Norm",       "#B4E0FF")
box(0.5, 3.0, 3.0, 1.0, "Feed-Forward + Add & Norm",                     "#C8E6C9")
box(0.5, 4.3, 3.0, 0.5, "× N (= 6) encoder layers",                      "#FFFFFF")
box(0.5, 5.0, 3.0, 0.8, "Encoder output  (B, src_len, d_model)",         "#FFF59D")
arrow(2.0, 1.2, 2.0, 1.6)
arrow(2.0, 2.8, 2.0, 3.0)
arrow(2.0, 4.0, 2.0, 5.0)

# Decoder stack (right)
box(6.5, 0.4, 3.0, 0.8, "Target (shifted)  →  Embedding + Pos. Enc.", "#FFE5B4")
box(6.5, 1.6, 3.0, 1.2, "Masked Multi-Head Self-Attn\n+ Add & Norm", "#B4E0FF")
box(6.5, 3.0, 3.0, 1.2, "Cross-Attention (Q=dec, K,V=enc)\n+ Add & Norm", "#F8BBD0")
box(6.5, 4.4, 3.0, 1.0, "Feed-Forward + Add & Norm",                  "#C8E6C9")
box(6.5, 5.6, 3.0, 0.5, "× N (= 6) decoder layers",                   "#FFFFFF")
box(6.5, 6.3, 3.0, 0.8, "Linear → Softmax  (next-token probs)",       "#FFAB91")
arrow(8.0, 1.2, 8.0, 1.6)
arrow(8.0, 2.8, 8.0, 3.0)
arrow(8.0, 4.2, 8.0, 4.4)
arrow(8.0, 5.4, 8.0, 6.3)

# Cross-attention arrow from encoder output to decoder cross-attn
arrow(3.5, 5.4, 6.5, 3.6, color="#C2185B")
ax.text(5.0, 4.7, "K, V\n(encoder output)", ha="center", va="center",
        fontsize=8, color="#C2185B", fontweight="bold")

ax.set_title("Figure 1 — Transformer (encoder ↔ decoder with cross-attention)",
             fontsize=12, fontweight="bold", pad=12)
plt.tight_layout(); plt.show()


### Figure 2 — Positional Encoding (sinusoidal pattern)

Each row is a position, each column is a dimension. Even dims use **sin**, odd dims use **cos**.
The frequency decreases with depth, so the model can distinguish absolute positions and learn relative offsets.


In [ ]:

import numpy as np
import math
import matplotlib.pyplot as plt

d_model_v = 128
max_len_v = 100
pe = np.zeros((max_len_v, d_model_v))
position = np.arange(0, max_len_v).reshape(-1, 1)
div_term = np.exp(np.arange(0, d_model_v, 2) * -(math.log(10000.0) / d_model_v))
pe[:, 0::2] = np.sin(position * div_term)
pe[:, 1::2] = np.cos(position * div_term)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im = axes[0].imshow(pe, aspect="auto", cmap="RdBu_r", origin="lower")
axes[0].set_xlabel("Dimension"); axes[0].set_ylabel("Position")
axes[0].set_title("PE matrix  (100 positions × 128 dims)")
fig.colorbar(im, ax=axes[0])

for dim in [4, 12, 32, 64]:
    axes[1].plot(pe[:, dim], label=f"dim {dim}")
axes[1].set_xlabel("Position"); axes[1].set_ylabel("PE value")
axes[1].set_title("Selected dimensions  (lower dim = higher freq.)")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle("Figure 2 — Sinusoidal Positional Encoding", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()


### Figure 3 — Attention Masks

- **Causal mask** stops the decoder from peeking at future tokens (autoregressive).
- **Padding mask** zeros out positions where the input is `<pad>`.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

T = 8
causal = np.tril(np.ones((T, T)))

src_len = 10
pad_mask = np.ones((1, src_len))
pad_mask[0, 7:] = 0  # last 3 positions are <pad>

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].imshow(causal, cmap="Greens", vmin=0, vmax=1)
axes[0].set_title("Causal (look-ahead) mask\nlower-triangular → attend only to past + self")
axes[0].set_xlabel("Key position"); axes[0].set_ylabel("Query position")
for i in range(T):
    for j in range(T):
        axes[0].text(j, i, int(causal[i, j]), ha="center", va="center",
                     color="black" if causal[i, j] else "gray", fontsize=9)

axes[1].imshow(pad_mask, cmap="Blues", vmin=0, vmax=1, aspect="auto")
axes[1].set_title("Padding mask\n0 = <pad> token (ignored by attention)")
axes[1].set_xlabel("Source position"); axes[1].set_yticks([])
for j in range(src_len):
    axes[1].text(j, 0, int(pad_mask[0, j]), ha="center", va="center",
                 color="white" if pad_mask[0, j] else "black", fontsize=10)

plt.suptitle("Figure 3 — Mask types used in the Transformer", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()


### Figure 4 — Scaled Dot-Product Attention

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

The heatmap below shows attention weights (rows = queries, cols = keys). Each row sums to 1.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)
src_tokens = ["the", "cat", "sat", "on", "the", "mat", ".", "<eos>"]
tgt_tokens = ["die", "katze", "saß", "auf", "der", "matte", ".", "<eos>"]
Tq, Tk = len(tgt_tokens), len(src_tokens)
d_k = 64

Q = np.random.randn(Tq, d_k)
K = np.random.randn(Tk, d_k)

# Bias the weights towards a diagonal alignment (illustrative)
for i in range(Tq):
    K[min(i, Tk-1)] += Q[i] * 1.4

scores = Q @ K.T / np.sqrt(d_k)
attn = np.exp(scores - scores.max(axis=1, keepdims=True))
attn = attn / attn.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im0 = axes[0].imshow(scores, cmap="coolwarm", aspect="auto")
axes[0].set_xticks(range(Tk)); axes[0].set_xticklabels(src_tokens, rotation=45, ha="right")
axes[0].set_yticks(range(Tq)); axes[0].set_yticklabels(tgt_tokens)
axes[0].set_title("Raw scores  QKᵀ / √dₖ")
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(attn, cmap="viridis", aspect="auto", vmin=0, vmax=attn.max())
axes[1].set_xticks(range(Tk)); axes[1].set_xticklabels(src_tokens, rotation=45, ha="right")
axes[1].set_yticks(range(Tq)); axes[1].set_yticklabels(tgt_tokens)
axes[1].set_title("Attention weights  softmax(scores)\n(rows sum to 1)")
fig.colorbar(im1, ax=axes[1])

plt.suptitle("Figure 4 — Scaled Dot-Product Attention  (EN → DE cross-attention)",
             fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()


### Figure 5 — Multi-Head Attention split

A single `d_model = 512` representation is split into `h = 8` heads of size `d_k = 64`. Each head learns a different relation (syntactic, positional, semantic), then heads are concatenated and projected.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

d_model, h = 512, 8
d_k = d_model // h
colors = plt.cm.tab10(np.linspace(0, 1, h))

fig, ax = plt.subplots(figsize=(13, 3.2))
ax.set_xlim(0, d_model); ax.set_ylim(0, 1); ax.axis("off")

ax.add_patch(plt.Rectangle((0, 0.55), d_model, 0.35, facecolor="#E0E0E0", edgecolor="black"))
ax.text(d_model/2, 0.73, f"d_model = {d_model}  (single Q / K / V projection)",
        ha="center", va="center", fontsize=10, fontweight="bold")

for i in range(h):
    x = i * d_k
    ax.add_patch(plt.Rectangle((x, 0.05), d_k, 0.35, facecolor=colors[i], edgecolor="black"))
    ax.text(x + d_k/2, 0.225, f"head {i}\nd_k={d_k}", ha="center", va="center", fontsize=8)

for i in range(h):
    x = i * d_k + d_k/2
    ax.annotate("", xy=(x, 0.42), xytext=(x, 0.53),
                arrowprops=dict(arrowstyle="->", lw=1.2, color="gray"))

ax.set_title("Figure 5 — Multi-head split  (d_model → h × d_k)",
             fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()


### Figure 6 — Teacher Forcing during Training

During training the decoder **always sees the ground-truth previous tokens** (not its own predictions). At step *t* it predicts token *t* given `<sos> y₁ … y_{t-1}`.


In [ ]:

import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

tgt_in  = ["<sos>", "die", "katze", "saß", "auf", "der", "matte", "."]
tgt_out = ["die",   "katze", "saß", "auf", "der", "matte", ".", "<eos>"]

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.set_xlim(-0.5, len(tgt_in)+0.5); ax.set_ylim(0, 4.2); ax.axis("off")

for i, tok in enumerate(tgt_in):
    ax.add_patch(plt.Rectangle((i, 2.6), 0.9, 0.7, facecolor="#B4E0FF", edgecolor="black"))
    ax.text(i + 0.45, 2.95, tok, ha="center", va="center", fontsize=10, fontweight="bold")

for i, tok in enumerate(tgt_out):
    ax.add_patch(plt.Rectangle((i, 0.3), 0.9, 0.7, facecolor="#FFAB91", edgecolor="black"))
    ax.text(i + 0.45, 0.65, tok, ha="center", va="center", fontsize=10, fontweight="bold")

for i in range(len(tgt_in)):
    ax.add_patch(FancyArrowPatch((i + 0.45, 2.6), (i + 0.45, 1.0),
                                 arrowstyle="->", mutation_scale=14, color="gray"))

ax.text(-0.5, 2.95, "Decoder input\n(shifted right)", ha="right", va="center",
        fontsize=10, fontweight="bold", color="#1565C0")
ax.text(-0.5, 0.65, "Loss target\n(next token)", ha="right", va="center",
        fontsize=10, fontweight="bold", color="#BF360C")

ax.set_title("Figure 6 — Teacher Forcing  (input is ground truth, target is shifted by one)",
             fontsize=12, fontweight="bold", pad=10)
plt.tight_layout(); plt.show()


### Figure 7 — Greedy Search vs Beam Search

**Greedy** picks the single best token at every step (fast, can lock into a bad prefix).
**Beam search** keeps the top-*k* hypotheses at each step and expands all of them (better quality, k× more compute).


In [ ]:

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Greedy path
ax = axes[0]
ax.set_xlim(0, 5); ax.set_ylim(-1, 5); ax.axis("off")
positions = [(0, 2), (1, 2), (2, 2.5), (3, 2.5), (4, 2)]
labels = ["<sos>", "die", "katze", "saß", "auf"]
for (x, y), lbl in zip(positions, labels):
    ax.add_patch(plt.Circle((x, y), 0.25, color="#FFAB91", ec="black"))
    ax.text(x, y, lbl, ha="center", va="center", fontsize=8, fontweight="bold")
for (x1, y1), (x2, y2) in zip(positions[:-1], positions[1:]):
    ax.annotate("", xy=(x2-0.25, y2), xytext=(x1+0.25, y1),
                arrowprops=dict(arrowstyle="->", lw=1.5))
ax.set_title("Greedy search\n(argmax at each step)", fontsize=11, fontweight="bold")

# Beam search tree
ax = axes[1]
ax.set_xlim(0, 5); ax.set_ylim(-1, 5); ax.axis("off")

root = (0, 2)
ax.add_patch(plt.Circle(root, 0.22, color="#FFAB91", ec="black"))
ax.text(root[0], root[1], "<sos>", ha="center", va="center", fontsize=7, fontweight="bold")

# Layer 1: 3 candidates, pick top-2
l1 = [(1, 3.2, "die"), (1, 2.0, "der"), (1, 0.8, "das")]
keep1 = {0, 1}
for i, (x, y, lbl) in enumerate(l1):
    c = "#FFAB91" if i in keep1 else "#EEEEEE"
    ax.add_patch(plt.Circle((x, y), 0.22, color=c, ec="black"))
    ax.text(x, y, lbl, ha="center", va="center", fontsize=7, fontweight="bold")
    ax.annotate("", xy=(x-0.22, y), xytext=(root[0]+0.22, root[1]),
                arrowprops=dict(arrowstyle="->", lw=1.2,
                                color="black" if i in keep1 else "lightgray"))

# Layer 2: from each kept beam expand 2
l2 = [(2, 3.6, "katze"), (2, 2.8, "hund"),
      (2, 2.2, "katze"), (2, 1.4, "tisch")]
keep2 = {0, 2}
parents = [0, 0, 1, 1]
for i, ((x, y, lbl), p) in enumerate(zip(l2, parents)):
    c = "#FFAB91" if i in keep2 else "#EEEEEE"
    ax.add_patch(plt.Circle((x, y), 0.22, color=c, ec="black"))
    ax.text(x, y, lbl, ha="center", va="center", fontsize=7, fontweight="bold")
    px, py, _ = l1[p]
    ax.annotate("", xy=(x-0.22, y), xytext=(px+0.22, py),
                arrowprops=dict(arrowstyle="->", lw=1.2,
                                color="black" if i in keep2 else "lightgray"))

ax.set_title("Beam search  (beam_width = 2)\nkeep top-2 hypotheses at every step",
             fontsize=11, fontweight="bold")

plt.suptitle("Figure 7 — Decoding strategies", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()


### Figure 8 — Token → Embedding → Output Flow

End-to-end data flow with shape annotations. Tokens become integer IDs, embeddings, positional-encoded vectors, contextual representations through the stacks, logits, and finally a probability distribution over the vocabulary.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

np.random.seed(0)

tokens = ["<sos>", "the", "cat", "sat", "<eos>"]
ids    = [2,        47,    312,   89,    3]
T = len(tokens)
d_model_v = 16   # tiny for visualization
V = 12           # tiny vocab for the output bar chart

# 1. Embedding lookup (random for illustration)
emb_table = np.random.randn(1000, d_model_v) * 0.5
embeddings = emb_table[ids]

# 2. Positional encoding
import math
pe = np.zeros((T, d_model_v))
pos = np.arange(T).reshape(-1, 1)
div = np.exp(np.arange(0, d_model_v, 2) * -(math.log(10000.0) / d_model_v))
pe[:, 0::2] = np.sin(pos * div); pe[:, 1::2] = np.cos(pos * div)
input_repr = embeddings + pe

# 3. Pretend the stack produces contextual reps
context = input_repr + np.random.randn(*input_repr.shape) * 0.3

# 4. Logits for the LAST position (predicting next token)
W_out = np.random.randn(d_model_v, V) * 0.4
logits = context[-1] @ W_out
probs = np.exp(logits - logits.max()); probs = probs / probs.sum()
vocab_labels = ["<pad>", "<unk>", "<sos>", "<eos>", "the", "a", "cat",
                "dog", "sat", "ran", "mat", "."]

# ---- plot ----
fig = plt.figure(figsize=(15, 11))
gs = fig.add_gridspec(5, 3, height_ratios=[0.6, 1.0, 1.0, 1.0, 1.4], hspace=0.55, wspace=0.25)

# Row 1 — token IDs
ax0 = fig.add_subplot(gs[0, :]); ax0.axis("off")
ax0.set_xlim(0, T); ax0.set_ylim(0, 1)
for i, (tok, tid) in enumerate(zip(tokens, ids)):
    ax0.add_patch(FancyBboxPatch((i+0.05, 0.15), 0.9, 0.7, boxstyle="round,pad=0.05",
                                  facecolor="#FFE0B2", edgecolor="black", linewidth=1.4))
    ax0.text(i+0.5, 0.62, tok, ha="center", va="center", fontsize=11, fontweight="bold")
    ax0.text(i+0.5, 0.32, f"id={tid}", ha="center", va="center", fontsize=9, color="#6D4C41")
ax0.text(-0.05, 0.5, "Step 1\nTokens\n(B,T)", ha="right", va="center",
         fontsize=9, fontweight="bold")

# Row 2 — embeddings heatmap
ax1 = fig.add_subplot(gs[1, :])
im1 = ax1.imshow(embeddings, aspect="auto", cmap="RdBu_r")
ax1.set_yticks(range(T)); ax1.set_yticklabels(tokens)
ax1.set_xlabel(f"d_model = {d_model_v}  (here tiny; real is 512)")
ax1.set_title(f"Step 2 — Embedding lookup  →  shape (B, T={T}, d_model={d_model_v})", fontsize=10, fontweight="bold")
fig.colorbar(im1, ax=ax1, fraction=0.02)

# Row 3 — + positional encoding
ax2 = fig.add_subplot(gs[2, :])
im2 = ax2.imshow(input_repr, aspect="auto", cmap="RdBu_r")
ax2.set_yticks(range(T)); ax2.set_yticklabels(tokens)
ax2.set_xlabel("d_model")
ax2.set_title("Step 3 — + Positional Encoding  →  encoder/decoder input", fontsize=10, fontweight="bold")
fig.colorbar(im2, ax=ax2, fraction=0.02)

# Row 4 — context after stack
ax3 = fig.add_subplot(gs[3, :])
im3 = ax3.imshow(context, aspect="auto", cmap="viridis")
ax3.set_yticks(range(T)); ax3.set_yticklabels(tokens)
ax3.set_xlabel("d_model")
ax3.set_title("Step 4 — After N encoder/decoder layers  →  contextual representation", fontsize=10, fontweight="bold")
fig.colorbar(im3, ax=ax3, fraction=0.02)

# Row 5 — logits + softmax for the last position
ax4a = fig.add_subplot(gs[4, 0])
ax4a.barh(range(V), logits, color="#90CAF9", edgecolor="black")
ax4a.set_yticks(range(V)); ax4a.set_yticklabels(vocab_labels, fontsize=8)
ax4a.set_xlabel("Logits"); ax4a.invert_yaxis()
ax4a.set_title("Step 5a — Linear  (d_model → vocab_size)", fontsize=10, fontweight="bold")
ax4a.grid(True, axis="x", alpha=0.3)

ax4b = fig.add_subplot(gs[4, 1])
top_idx = np.argmax(probs)
colors_v = ["#EF5350" if i == top_idx else "#A5D6A7" for i in range(V)]
ax4b.barh(range(V), probs, color=colors_v, edgecolor="black")
ax4b.set_yticks(range(V)); ax4b.set_yticklabels(vocab_labels, fontsize=8)
ax4b.set_xlabel("Probability"); ax4b.invert_yaxis()
ax4b.set_title("Step 5b — Softmax  →  next-token P(y | ...)", fontsize=10, fontweight="bold")
ax4b.grid(True, axis="x", alpha=0.3)

ax4c = fig.add_subplot(gs[4, 2]); ax4c.axis("off")
top3 = np.argsort(-probs)[:3]
ax4c.text(0.0, 0.95, "Step 6 — Decoding choice", fontsize=11, fontweight="bold",
          transform=ax4c.transAxes)
ax4c.text(0.0, 0.78, "Greedy   →  argmax", transform=ax4c.transAxes, fontsize=10, color="#C62828")
ax4c.text(0.05, 0.68, f"pick: '{vocab_labels[top3[0]]}'  (p={probs[top3[0]]:.2f})",
          transform=ax4c.transAxes, fontsize=10)
ax4c.text(0.0, 0.50, "Beam (k=3)  →  keep top-k", transform=ax4c.transAxes, fontsize=10, color="#1565C0")
for j, t in enumerate(top3):
    ax4c.text(0.05, 0.40 - j*0.08, f"{j+1}. '{vocab_labels[t]}'  (p={probs[t]:.2f})",
              transform=ax4c.transAxes, fontsize=10)

fig.suptitle("Figure 8 — End-to-end data flow:  token IDs → embeddings → encoder/decoder → logits → next-token distribution",
             fontsize=12, fontweight="bold", y=0.995)
plt.show()

print("Shapes at each stage:")
print(f"  Step 1  tokens         : (B=1, T={T})")
print(f"  Step 2  embeddings     : (B=1, T={T}, d_model={d_model_v})")
print(f"  Step 3  +PE            : (B=1, T={T}, d_model={d_model_v})")
print(f"  Step 4  context        : (B=1, T={T}, d_model={d_model_v})")
print(f"  Step 5  logits         : (B=1, T={T}, vocab={V})")
print(f"  Step 5  probs (last t) : (vocab={V},)   sums to {probs.sum():.4f}")


## 9. Smoke Test — Run the Model on a Synthetic Batch

Verifies the entire model wires up correctly without needing to download Multi30k.

In [ ]:
torch.manual_seed(0)

src_vocab_size = 10000
tgt_vocab_size = 10000

model = Transformer(
    src_vocab_size=src_vocab_size,
    tgt_vocab_size=tgt_vocab_size,
    d_model=512,
    num_encoder_layers=6,
    num_decoder_layers=6,
    num_heads=8,
    d_ff=2048,
    max_seq_length=100,
    dropout=0.1,
    pad_idx=0,
)

batch_size, src_len, tgt_len = 2, 10, 8
src = torch.randint(1, src_vocab_size, (batch_size, src_len))
tgt = torch.randint(1, tgt_vocab_size, (batch_size, tgt_len))

out = model(src, tgt)
print(f'src       : {tuple(src.shape)}')
print(f'tgt       : {tuple(tgt.shape)}')
print(f'output    : {tuple(out.shape)}  (expected ({batch_size}, {tgt_len}, {tgt_vocab_size}))')
total = sum(p.numel() for p in model.parameters())
print(f'parameters: {total:,}')

## 10. End-to-End Run (requires Multi30k download)

Uncomment to train on Multi30k. Expect ~20 epochs on GPU. On CPU, lower `batch_size`, `num_epochs`, and slice the dataset.

In [ ]:
# # 1. Load + preprocess data
# src_train, tgt_train = load_and_preprocess_multi30k('train', 'en', 'de')
# src_val,   tgt_val   = load_and_preprocess_multi30k('validation', 'en', 'de')
# src_test,  tgt_test  = load_and_preprocess_multi30k('test', 'en', 'de')
#
# # 2. Train tokenizers
# tt = BilingualTokenizerTrainer(vocab_size=10000, min_frequency=2)
# shared_tok = tt.train_shared_tokenizer(src_train, tgt_train, './tokenizers/shared_tokenizer.json')
# src_tokenizer = shared_tok
# tgt_tokenizer = shared_tok
#
# # 3. Dataloaders
# train_loader, val_loader = create_dataloaders(
#     src_train, tgt_train, src_val, tgt_val,
#     src_tokenizer, tgt_tokenizer,
#     batch_size=config.batch_size, max_length=config.max_seq_length, num_workers=0,
# )
#
# # 4. Model
# vocab_size = len(src_tokenizer.get_vocab())
# pad_idx = src_tokenizer.token_to_id('<pad>')
# device = config.get_device()
# model = Transformer(
#     src_vocab_size=vocab_size, tgt_vocab_size=vocab_size,
#     d_model=config.d_model, num_encoder_layers=config.num_encoder_layers,
#     num_decoder_layers=config.num_decoder_layers, num_heads=config.num_heads,
#     d_ff=config.d_ff, max_seq_length=config.max_seq_length,
#     dropout=config.dropout, pad_idx=pad_idx,
# )
#
# # 5. Optimizer + scheduler + loss
# optimizer = optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
# scheduler = NoamScheduler(optimizer, d_model=config.d_model, warmup_steps=config.warmup_steps)
# criterion = LabelSmoothingLoss(vocab_size=vocab_size, pad_idx=pad_idx, smoothing=config.label_smoothing)
#
# # 6. Train
# trainer = Trainer(model, train_loader, val_loader, optimizer, scheduler, criterion, device=device)
# trainer.train(num_epochs=config.num_epochs)
#
# # 7. Evaluate
# generator = TranslationGenerator(model, src_tokenizer, tgt_tokenizer, device=device)
# hyps = generator.translate_batch(src_test[:100], method='beam', beam_width=5)
# print('BLEU:', compute_bleu(hyps, tgt_test[:100]))
#
# # 8. Visualize
# plot_training_history('./checkpoints/training_history.json')